# Aethera · reconstrucción del diagnóstico y evaluación del prototipo

**Entregable 2 · Desafío Ciudad Aethera · Misión M5**

Este notebook deja trazable todo lo que el E1 afirmó y todo lo que el E2 midió.
Si una cifra se cuestiona, se ejecuta esto y se reproduce.

> Los datos son 100 % sintéticos del Data Pack de la organización. Aethera es una ciudad
> universitaria ficticia: no hay personas, lugares ni servicios reales.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from aethera import config as C, datos, evaluacion, horarios, ml, segmentos

pd.set_option("display.width", 200)
print("Entorno listo.")

Entorno listo.


## 1 · El Data Pack

In [2]:
for k, v in datos.resumen().items():
    print(f"{k:12} {v}")

D1           12000
D2           9400
D3           130000
D4           6000
D5           2500
D6           15
D7           32
espera_min   4
espera_max   120
espera_media 42.0


## 2 · Reconstrucción de M5 (Entregable 1)

El embudo debe dar exactamente las cifras del E1. Si no coinciden, algo se rompió.

In [3]:
emb = segmentos.segmentacion().embudo
display(pd.DataFrame([emb]).T.rename(columns={0: "valor"}))

,valor
institucion,UNI_NOVA_AETHER
encuestados,6697
moderada_alta,2739
m5,1868
m5_pct,0.682001
prioritario,826
prioritario_pct_m5,0.442184
prioritario_pct_mod,0.30157


## 3 · Las fricciones que explican la brecha

Tres diferencias grandes, una descartada y una degradada a hipótesis por causalidad inversa.
Es la parte del E1 que justifica el diseño del producto.

In [4]:
nova = segmentos.segmentacion().nova
mod = nova[nova["es_mod_alta"]]

def brecha(condicion, etiqueta):
    con, sin = mod[condicion], mod[~condicion]
    return {"fricción": etiqueta,
            "con la fricción": con["previous_support_use"].eq("never").mean(),
            "sin la fricción": sin["previous_support_use"].eq("never").mean()}

tabla = pd.DataFrame([
    brecha(nova["f_desconoce"], "desconoce los servicios"),
    brecha(nova["f_sin_red"], "sin red de apoyo"),
    brecha(nova["f_nocturno"], "horario nocturno"),
    brecha(nova["campus_activity_frequency"].isin(["low", "none"]), "poca participación en campus"),
    brecha(nova["evaluation_overload"].eq("yes"), "sobrecarga de evaluaciones (descartada)"),
])
tabla["diferencia (pp)"] = (tabla["con la fricción"] - tabla["sin la fricción"]) * 100
tabla.iloc[:, 1:] = tabla.iloc[:, 1:].round(3)
display(tabla)

,fricción,con la fricción,sin la fricción,diferencia (pp)
0,desconoce los servicios,0.807,0.585,22.249
1,sin red de apoyo,0.733,0.568,16.439
2,horario nocturno,0.776,0.671,10.565
3,poca participación en campus,0.734,0.635,9.951
4,sobrecarga de evaluaciones (descartada),0.681,0.685,-0.361


## 4 · El efecto de acumular fricciones

Con 0 fricciones casi la mitad de M5 ya es alcanzable. Con 3, la brecha es casi total.

In [5]:
mod2 = mod.copy()
mod2["n_fricciones"] = mod2[["f_desconoce", "f_sin_red", "f_nocturno"]].sum(axis=1)
g = mod2.groupby("n_fricciones").agg(
    estudiantes=("student_id", "size"),
    nunca_busco_ayuda=("previous_support_use", lambda s: s.eq("never").mean()),
).round(3)
g["no_busco_ayuda"] = g["nunca_busco_ayuda"].map("{:.1%}".format)
g = g.drop(columns="nunca_busco_ayuda").rename(columns={"no_busco_ayuda": "no buscó apoyo"})
display(g)

,estudiantes,no buscó apoyo
n_fricciones,,
0,441,47.2%
1,1318,63.3%
2,869,83.7%
3,111,89.2%


## 5 · Sesgos del Data Pack (oleada 2: D4 y D5)

Tres sesgos deliberados que el enunciado revela después del Demo Day. Los detectamos antes,
y cada uno es la razón de una decisión de diseño.

In [6]:
d = datos.cargar()

print("SESGO 1 · el sentimiento de D4 refleja el canal, no la persona")
sent = pd.crosstab(d["D4"]["source_type"], d["D4"]["sentiment"], normalize="index")
display(sent.round(2))

print("\nSESGO 2 · el motivo declarado en D5 no predice el texto")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
acc = cross_val_score(LogisticRegression(max_iter=2000),
                      TfidfVectorizer().fit_transform(d["D4"]["text"]),
                      d["D4"]["topic"], cv=5).mean()
mayor = d["D4"]["topic"].value_counts(normalize=True).max()
print(f"  clasificador de texto: {acc:.1%}   clase mayoritaria: {mayor:.1%}   "
      f"→ sin señal útil, no entrenamos con D4")

print("\nSESGO 3 · la derivación de D5 no depende del motivo")
coh = (d["D5"]["tipo_derivado"] == d["D5"]["tipo_esperado"]).mean()
print(f"  coherencia con el mapa de D2: {coh:.1%}   azar entre 3 tipos: 33.3%")
print(f"  fuera de lun–vie 09:00–18:00: {((d['D5']['inicio'].dt.hour < 9) | (d['D5']['inicio'].dt.hour >= 18) | (d['D5']['inicio'].dt.weekday >= 5)).mean():.0%}")

SESGO 1 · el sentimiento de D4 refleja el canal, no la persona


sentiment,hopeful,mixed,strained
source_type,,,
student_forum,0.5,0.5,0.0
wellbeing_mailbox,0.0,0.0,1.0



SESGO 2 · el motivo declarado en D5 no predice el texto


  clasificador de texto: 40.9%   clase mayoritaria: 43.2%   → sin señal útil, no entrenamos con D4

SESGO 3 · la derivación de D5 no depende del motivo
  coherencia con el mapa de D2: 33.8%   azar entre 3 tipos: 33.3%
  fuera de lun–vie 09:00–18:00: 73%


## 6 · El modelo de asistencia y su límite

Regresión logística sobre D2. Se declara el comportamiento en la franja que la demo ofrece.

In [7]:
m = ml.modelo()
print("Comparativa de modelos")
display(m.comparativa)
print("\nImportancia de features (AUC)")
display(m.importancia)

print("\nEfecto de la espera y confianza asociada")
for e in [4, 14, 20, 39, 42, 70, 120]:
    p = ml.p_asistencia("digital", "counseling", "academic_pressure", e)
    c, nota = m.confianza(e)
    print(f"  {e:3} días  P={p:.3f}  confianza={c:.2f}  ({nota})")

Comparativa de modelos


,modelo,AUC,Brier,lift_vs_azar
0,Regresión logística,0.5665,0.206,0.0665
1,Gradient boosting,0.5421,0.209,0.0421



Importancia de features (AUC)


,feature,importancia
0,espera_dias,0.0530
1,contact_channel,0.0199
2,reason_code,0.0041
3,service_type,0.0019



Efecto de la espera y confianza asociada
    4 días  P=0.793  confianza=0.07  (extrapolación: casi sin datos para esta espera)
   14 días  P=0.772  confianza=0.60  (con poca evidencia histórica)
   20 días  P=0.760  confianza=1.00  (dentro del rango observado)
   39 días  P=0.716  confianza=1.00  (dentro del rango observado)
   42 días  P=0.709  confianza=1.00  (dentro del rango observado)
   70 días  P=0.636  confianza=1.00  (dentro del rango observado)
  120 días  P=0.491  confianza=0.84  (dentro del rango observado)


## 7 · El clasificador de necesidad entrenado en D5

Sustituye a las palabras clave escritas a mano, que rendían 27 %: peor que la clase mayoritaria.

In [8]:
c = ml.entrenar_clasificador_texto()
print(f"  Accuracy (CV 5 folds)  {c['accuracy_cv']:.1%}")
print(f"  Línea base             {c['linea_base']:.1%}")
print(f"  n conversaciones       {c['n']:,}")

for t in ["Tengo tres parciales y no sé por dónde empezar",
          "Desde que me mudé no tengo con quién hablar",
          "Duermo 4 horas y ando agotado todo el día"]:
    print(f"    {t[:44]:46} → {ml.clasificar_texto(t)[0]}")

  Accuracy (CV 5 folds)  80.8%
  Línea base             20.6%
  n conversaciones       2,500
    Tengo tres parciales y no sé por dónde empez   → preventive_guidance
    Desde que me mudé no tengo con quién hablar    → career_concern
    Duermo 4 horas y ando agotado todo el día      → academic_pressure


## 8 · La fricción que el producto elimina

300 estudiantes reales del segmento prioritario contra un directorio estático.

In [9]:
b = evaluacion.comparar_ruta_directorio(n=300)
for k in ["n", "turnos_promedio_por_estudiante", "fraccion_turnos_compatibles",
          "mediana_turnos_compatibles_por_estudiante", "directorio_con_alguno_turno_util",
          "ruta_match_rate", "dias_hasta_primera_opcion"]:
    print(f"  {k:42} {b[k]}")
print(f"\n  {b['nota']}")

  n                                          300
  turnos_promedio_por_estudiante             370.2
  fraccion_turnos_compatibles                0.2815595173779939
  mediana_turnos_compatibles_por_estudiante  0.27696793002915454
  directorio_con_alguno_turno_util           0.8733333333333333
  ruta_match_rate                            1.0
  dias_hasta_primera_opcion                  13.69

  El 100 % de Ruta es por construccion: filtrar la agenda ES el producto. La cifra util es cuantos turnos tenia que revisar el estudiante para encontrar uno viable.


## 9 · Proyección sobre el indicador de la misión (68 %)

In [10]:
p = evaluacion.proyectar_impacto()
print(f"  Denominador (ansiedad moderada/alta)   {p['moderada_alta']:,}")
print(f"  Línea base                            {p['m5_pct_base']:.1%}")
print(f"  Proyectado (30% x 35%)                {p['m5_pct_proyectado']:.1%}  ({p['reduccion_pp']:+.1f} pp)")
for s in p["supuestos"]:
    print(f"    · {s}")
display(evaluacion.sensibilidad())

  Denominador (ansiedad moderada/alta)   2,739
  Línea base                            68.2%
  Proyectado (30% x 35%)                65.0%  (+3.2 pp)
    · Denominador: 2,739 estudiantes con ansiedad moderada/alta (igual que la línea base del 68 %).
    · Alcance 30%: fracción del segmento prioritario (826) que llega a usar Ruta.
    · Conversión 35%: de quienes usan Ruta, fracción que completa una reserva.
    · Contrafactual conservador: se asume que sin Ruta esa demanda no se materializa. No está demostrado.
    · No se modela efecto sobre los 1,042 estudiantes de M5 fuera del segmento prioritario.


,alcance,conversión,M5 antes,M5 después,reducción
0,10%,20%,68.2%,67.6%,+0.6 pp
1,10%,35%,68.2%,67.1%,+1.1 pp
2,10%,50%,68.2%,66.7%,+1.5 pp
3,20%,20%,68.2%,67.0%,+1.2 pp
4,20%,35%,68.2%,66.1%,+2.1 pp
5,20%,50%,68.2%,65.2%,+3.0 pp
6,30%,20%,68.2%,66.4%,+1.8 pp
7,30%,35%,68.2%,65.0%,+3.2 pp
8,30%,50%,68.2%,63.7%,+4.5 pp
9,50%,20%,68.2%,65.2%,+3.0 pp


---

## Cómo reproducirlo

```bash
pip install -r requirements.txt
jupyter notebook notebook/ENTREGABLE2_analisis.ipynb
```

La agenda usa semilla fija, así que los resultados son idénticos en cada ejecución.
Las mismas cifras se obtienen sin notebook con:

```bash
python -m aethera.segmentos
python -m aethera.ml
python -m aethera.evaluacion
python tests/verificar_e2.py
```